# Phân tích chất lượng dữ liệu ban đầu (Data Quality)
Đánh giá mức độ khuyết thiếu, trùng lặp và tính nhất quán của bộ dữ liệu thô (Raw Data) để định hướng cho quá trình làm sạch.


In [8]:
import warnings
from pathlib import Path
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

## 1. Khởi tạo đường dẫn và nạp dữ liệu thô

In [4]:
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name in {"notebooks", "src"} else current_dir

# SỬA LỖI: Trỏ vào thư mục raw để lấy file cafeland_raw.csv
data_path = project_root / "data" / "raw" / "cafeland_raw.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Không tìm thấy file dữ liệu thô tại: {data_path}")

df = pd.read_csv(data_path, encoding="utf-8-sig")
print(f"Đã nạp thành công bộ dữ liệu THÔ: {len(df):,} dòng và {len(df.columns)} cột.")
display(df.head())

Đã nạp thành công bộ dữ liệu THÔ: 6,500 dòng và 8 cột.


,tieu_de,gia,dien_tich,vi_tri,ngay_dang,link_nguon,mo_ta,nguoi_ban
0,KẸT TIỀN CẦN BÁN 2PN+ 72M TẦNG 20+ MAISON GRAN...,2 tỷ 990 triệu,72m2,"Phú Mỹ, TP. Hồ Chí Minh",16-09-2026 09:17:35,https://nhadat.cafeland.vn/ket-tien-can-ban-2p...,**2PN+ 72M² TẦNG CAO – MAISON GRAND PHÚ MỸ | G...,Phạm Thị Quyên
1,"BÁN NHÀ MỚI KENG MẶT TIỀN TÂN XUÂN, HÓC MÔN ,S...",5 tỷ 350 triệu,100m2,"An Lạc, TP. Hồ Chí Minh",16-09-2026 09:06:35,https://nhadat.cafeland.vn/ban-nha-moi-keng-ma...,"BÁN NHÀ MỚI KENG MẶT TIỀN ĐƯỜNG KINH DOANH,ẤP ...",Trần Ngọc Tuấn
2,"ĐẤT 1/NGUYỄN THỊ SÁNG , ĐÔNG THẠNH ,HÓC MÔN 5M...",3 tỷ 750 triệu,105m2,"Chợ Lớn, TP. Hồ Chí Minh",16-09-2026 09:06:29,https://nhadat.cafeland.vn/dat-1nguyen-thi-san...,"HÀNG F0 MỚI RA LÒ , ĐẤT 1/NGUYỄN THỊ SÁNG , ĐÔ...",Trần Ngọc Tuấn
3,CHỦ CẦN BÁN GẤP GIẢM MẠNH 6 TỶ BÁN NHÀ MẶT TIỀ...,39 tỷ,454m2,"Hóc Môn, TP. Hồ Chí Minh",16-09-2026 09:06:24,https://nhadat.cafeland.vn/chu-can-ban-gap-gia...,CHỦ CẦN BÁN GẤP GIẢM MẠNH 6 TỶ\r\nBÁN NHÀ MẶT ...,Trần Ngọc Tuấn
4,BÁN ĐẤT MẶT TIỀN ĐƯỜNG NHỰA DIỆN TÍCH KHỦNG 1/...,18 tỷ,1270.3m2,"Hóc Môn, TP. Hồ Chí Minh",16-09-2026 09:06:18,https://nhadat.cafeland.vn/ban-dat-mat-tien-du...,BÁN ĐẤT MẶT TIỀN ĐƯỜNG NHỰA DIỆN TÍCH KHỦNG 1/...,Tuấnland88


## 2. Kiểm tra Dữ liệu thiếu (Missing Values) và Trùng lặp

In [13]:
total_rows = len(df)

# Biến tất cả các ô chỉ chứa khoảng trắng hoặc chuỗi rỗng thành giá trị rỗng chuẩn (NaN) của thư viện Numpy
import numpy as np
df_check = df.replace(r'^\s*$', np.nan, regex=True)

# 1. Đánh giá dữ liệu thiếu (Sử dụng df_check thay vì df)
missing = (df_check.isna().mean() * 100).round(2)
missing_df = pd.DataFrame({
    "Số bản ghi thiếu": df_check.isna().sum(),
    "Tỷ lệ thiếu (%)": missing,
}).sort_values(by="Tỷ lệ thiếu (%)", ascending=False)

print("--- TỶ LỆ DỮ LIỆU KHUYẾT THIẾU ---")
# Bỏ điều kiện > 0 để hiển thị toàn bộ các cột trong dataset
display(missing_df)

# 2. Kiểm tra trùng lặp toàn bộ dòng
exact_dups = df_check.duplicated().sum()
print("\n--- KIỂM TRA TRÙNG LẶP ---")
print(f"- Số dòng trùng lặp hoàn toàn: {exact_dups} dòng (Chiếm {exact_dups / total_rows * 100:.2f}%)")

# 3. Kiểm tra tính nhất quán (Dữ liệu rác)
print("\n--- KIỂM TRA TÍNH NHẤT QUÁN CƠ BẢN ---")
if "gia" in df_check.columns:
    count_thuong_luong = df_check['gia'].str.contains('Thương lượng|Thỏa thuận', case=False, na=False).sum()
    print(f"- Số tin đăng có giá 'Thương lượng' (cần xử lý thành NaN): {count_thuong_luong}")

--- TỶ LỆ DỮ LIỆU KHUYẾT THIẾU ---


,Số bản ghi thiếu,Tỷ lệ thiếu (%)
tieu_de,0,0.0
gia,0,0.0
dien_tich,0,0.0
vi_tri,0,0.0
ngay_dang,0,0.0
link_nguon,0,0.0
mo_ta,0,0.0
nguoi_ban,0,0.0



--- KIỂM TRA TRÙNG LẶP ---
- Số dòng trùng lặp hoàn toàn: 51 dòng (Chiếm 0.78%)

--- KIỂM TRA TÍNH NHẤT QUÁN CƠ BẢN ---
- Số tin đăng có giá 'Thương lượng' (cần xử lý thành NaN): 492
